In [1]:
import pandas as pd 
import os

In [ ]:

def load_csv(file_name):
    data_path = 'C:/Users/DELL/Projects/ecommerce-sales-pipeline/data/raw'
    raw=os.path.join(data_path,file_name)
    return pd.read_csv(raw,encoding='latin1', sep=";")

Location= load_csv('Location.csv')
Products = load_csv('Products.csv')
Orders= load_csv('Orders.csv')


#reusable csv loader to avoid repeating file-loading logic across datasets

In [ ]:


def convert_type(df,desired_type):
    column = df.astype(desired_type)
    return  column

Location['Postal Code']=convert_type(Location['Postal Code'],'str')
Orders['Postal Code']=convert_type(Orders['Postal Code'],'str')

# reusable type conversion to avoid repeating multiple type conversions across datasets

In [ ]:
def convert_datetype(date,dateFormat="%d/%m/%Y"):
    appropriate_dateformat = pd.to_datetime(date,format=dateFormat)

    return appropriate_dateformat


Orders['Ship Date'] = convert_datetype(Orders['Ship Date'])
Orders['Order Date'] = convert_datetype(Orders['Order Date'])


# reusable date conversion to avoid repeating multiple date conversions across datasets


In [13]:
Products[Products[['Category','Sub-Category','Product Name,,,,,']].isna().any(axis=1)]



,Product ID,Category,Sub-Category,"Product Name,,,,,"
26,"FUR-BO-10002916;Furniture;Bookcases;""Rush Hier...",NaN,NaN,NaN
142,"FUR-FU-10000087;Furniture;Furnishings;""Executi...",NaN,NaN,NaN
147,"FUR-FU-10000222;Furniture;Furnishings;""Seth Th...",NaN,NaN,NaN
149,"FUR-FU-10000260;Furniture;Furnishings;""6"""" Cub...",NaN,NaN,NaN
152,"FUR-FU-10000305;Furniture;Furnishings;""Tenex V...",NaN,NaN,NaN
...,...,...,...,...
1479,"OFF-SU-10004782;Office Supplies;Supplies;""Elit...",NaN,NaN,NaN
1615,"TEC-AC-10004659;Technology;Accessories;""Imatio...",NaN,NaN,NaN
1657,"TEC-MA-10001127;Technology;Machines;""HP Design...",NaN,NaN,NaN
1731,"TEC-PH-10000702;Technology;Phones;""Square Cred...",NaN,NaN,NaN


In [14]:

malformed_rows= (
    Products['Category'].isna() & 
    Products['Sub-Category'].isna() &
    Products['Product Name,,,,,'].isna())


Products.loc[malformed_rows,'Product ID'].str.split(';',expand=True)


,0,1,2,3,4
26,FUR-BO-10002916,Furniture,Bookcases,"""Rush Hierlooms Collection 1"""" Thick Stackable...",None
142,FUR-FU-10000087,Furniture,Furnishings,"""Executive Impressions 14"""" Two-Color Numerals...",None
147,FUR-FU-10000222,Furniture,Furnishings,"""Seth Thomas 16"""" Steel Case Clock"",,,,,",None
149,FUR-FU-10000260,Furniture,Furnishings,"""6"""" Cubicle Wall Clock,"" Black"""""",,,,",None
152,FUR-FU-10000305,Furniture,Furnishings,"""Tenex V2T-RE Standard Weight Series Chair Mat...",None
...,...,...,...,...,...
1479,OFF-SU-10004782,Office Supplies,Supplies,"""Elite 5"""" Scissors"",,,,,",None
1615,TEC-AC-10004659,Technology,Accessories,"""ImationSecure+ Hardware Encrypted USB 2.0Fl...","16GB"",,,,,"
1657,TEC-MA-10001127,Technology,Machines,"""HP Designjet T520 Inkjet Large Format Printer...",None
1731,TEC-PH-10000702,Technology,Phones,"""Square Credit Card Reader,"" 4 1/2"""""""" x 4 1/2...",None


In [15]:
Products.loc[malformed_rows,['Product ID', 'Category', 'Sub-Category', 'Product Name,,,,,']] = Products.loc[malformed_rows,'Product ID'].str.split(';', n=3, expand=True).values

In [16]:
Products.head()

,Product ID,Category,Sub-Category,"Product Name,,,,,"
0,FUR-BO-10000112,Furniture,Bookcases,"Bush Birmingham Collection Bookcase, Dark Cher..."
1,FUR-BO-10000330,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Plank..."
2,FUR-BO-10000362,Furniture,Bookcases,"Sauder Inglewood Library Bookcases,,,,,"
3,FUR-BO-10000468,Furniture,Bookcases,"O'Sullivan 2-Shelf Heavy-Duty Bookcases,,,,,"
4,FUR-BO-10000711,Furniture,Bookcases,"Hon Metal Bookcases, Gray,,,,"


In [17]:
Products['Product Name,,,,,'] = Products['Product Name,,,,,'].str.rstrip(',')
Products['Product Name'] = Products['Product Name,,,,,']

In [18]:
Products = Products.drop('Product Name,,,,,', axis=1)

In [19]:
Products.head()

,Product ID,Category,Sub-Category,Product Name
0,FUR-BO-10000112,Furniture,Bookcases,"Bush Birmingham Collection Bookcase, Dark Cherry"
1,FUR-BO-10000330,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Plank..."
2,FUR-BO-10000362,Furniture,Bookcases,Sauder Inglewood Library Bookcases
3,FUR-BO-10000468,Furniture,Bookcases,O'Sullivan 2-Shelf Heavy-Duty Bookcases
4,FUR-BO-10000711,Furniture,Bookcases,"Hon Metal Bookcases, Gray"


In [20]:
Products[Products['Product ID'].duplicated(keep=False)]



,Product ID,Category,Sub-Category,Product Name
18,FUR-BO-10002213,Furniture,Bookcases,DMI Eclipse Executive Suite Bookcases
19,FUR-BO-10002213,Furniture,Bookcases,"Sauder Forest Hills Library, Woodland Oak Finish"
66,FUR-CH-10001146,Furniture,Chairs,"Global Value Mid-Back Manager's Chair, Gray"
67,FUR-CH-10001146,Furniture,Chairs,"Global Task Chair, Black"
185,FUR-FU-10001473,Furniture,Furnishings,DAX Wood Document Frame
...,...,...,...,...
1789,TEC-PH-10002200,Technology,Phones,Aastra 6757i CT Wireless VoIP phone
1793,TEC-PH-10002310,Technology,Phones,Panasonic KX T7731-B Digital phone
1794,TEC-PH-10002310,Technology,Phones,Plantronics Calisto P620-M USB Wireless Speake...
1874,TEC-PH-10004531,Technology,Phones,OtterBox Commuter Series Case - iPhone 5 & 5s


In [22]:
def profile_col(df):
    profile = {
        "Data types": df.dtypes,
        "Missing values":df.isna().sum(),
        "Missing percentage": (df.isna().mean() * 100 ).round(2),
        "Unique values": df.nunique(),
        "duplicate":df.duplicated().sum()
    }
    return pd.DataFrame(profile)

profile_col(Orders)


,Data types,Missing values,Missing percentage,Unique values,duplicate
Row ID,int64,0,0.0,9994,0
Order ID,object,0,0.0,5009,0
Order Date,object,0,0.0,1236,0
Ship Date,object,0,0.0,1334,0
Ship Mode,object,0,0.0,4,0
Customer ID,object,0,0.0,793,0
Segment,object,0,0.0,3,0
Postal Code,int64,0,0.0,630,0
Product ID,object,0,0.0,1862,0
Sales,float64,0,0.0,5825,0


In [ ]:
# Orders['Ship Date']= pd.to_datetime(Orders['Ship Date'],format="%d/%m/%Y")
# Orders['Order Date']=pd.to_datetime(Orders['Order Date'],format="%d/%m/%Y")
# Orders['Postal Code'] = Orders['Postal Code'].astype('str')

In [24]:
Orders['Postal Code'] = Orders['Postal Code'].astype('str')

In [25]:
invalid_products= Orders[~Orders['Product ID'].isin(Products['Product ID'])]
print(invalid_products)

Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Segment, Postal Code, Product ID, Sales, Quantity, Discount, Profit]
Index: []


In [26]:
Orders.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,Postal Code,Product ID,Sales,Quantity,Discount,Profit
0,1,CA-2022-152156,2022-11-08,2022-11-11,Second Class,CG-12520,Consumer,42420,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,2,CA-2022-152156,2022-11-08,2022-11-11,Second Class,CG-12520,Consumer,42420,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,3,CA-2022-138688,2022-06-12,2022-06-16,Second Class,DV-13045,Corporate,90036,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,4,US-2021-108966,2021-10-11,2021-10-18,Standard Class,SO-20335,Consumer,33311,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,5,US-2021-108966,2021-10-11,2021-10-18,Standard Class,SO-20335,Consumer,33311,OFF-ST-10000760,22.3680,2,0.20,2.5164
